In [1]:
import requests
import json

In [99]:
id = "0htibyjstavahc"

In [100]:
# xxxxxxxxx-8000
url_id = f"https://{id}-8000.proxy.runpod.net/"

In [101]:
# 헬스 체크
health_check_url = f"{url_id}health"

In [102]:
health_check_res = requests.get(health_check_url)
if health_check_res.status_code != 200:
    print("헬스 체크 실패")
    exit()

print(health_check_res.json())


{'status': 'ok'}


In [103]:
# 모델 목록 
model_list_url = f"{url_id}v1/models"

In [104]:
model_list_res = requests.get(model_list_url)

if model_list_res.status_code != 200:
    print("모델 목록 조회 실패")
    exit()

model_list = model_list_res.json()

print(model_list)

model_name = model_list['data'][0]['id']

print(model_name)

{'object': 'list', 'data': [{'id': 'MiniMaxAI/SynLogic-32B', 'object': 'model', 'created': 1775443310, 'owned_by': 'vllm', 'root': 'MiniMaxAI/SynLogic-32B', 'parent': None, 'max_model_len': 32768, 'permission': [{'id': 'modelperm-8dae56cf7e3a9206', 'object': 'model_permission', 'created': 1775443310, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}
MiniMaxAI/SynLogic-32B


In [105]:
url = f"{url_id}v1/chat/completions"

In [106]:
payload = {
    "model": model_name,
    "messages": [
        {"role": "user", "content": "딸기 제철 시기는 언제야?"}
    ],
    "max_tokens": 4000,
    "temperature": 0,
    "stream": False
}

In [107]:
res = requests.post(url, json=payload)

In [108]:
print(res['choices'][0]['message']['content'])


TypeError: 'Response' object is not subscriptable

In [109]:
print(res.json())

{'id': 'chatcmpl-8ca1201a437567cb', 'object': 'chat.completion', 'created': 1775443318, 'model': 'MiniMaxAI/SynLogic-32B', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '딸기 제철은 보통 봄철인 4월부터 6월까지입니다. 이 기간 동안 딸기의 품질과 맛이 가장 좋습니다.', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning': None}, 'logprobs': None, 'finish_reason': 'stop', 'stop_reason': None, 'token_ids': None}], 'service_tier': None, 'system_fingerprint': None, 'usage': {'prompt_tokens': 28, 'total_tokens': 72, 'completion_tokens': 44, 'prompt_tokens_details': None}, 'prompt_logprobs': None, 'prompt_token_ids': None, 'kv_transfer_params': None}


In [112]:
payload = {
    "model": model_name,
    "messages": [
        {"role": "user", "content": "로또번호 추천기를 파이선코드로 작성해봐"}
    ],
    "temperature": 0,
    "stream": True,
    # qwen3 용
    "chat_template_kwargs": {"enable_thinking": False}
}

In [61]:
res = requests.post(url, json=payload)

for line in res.iter_lines():
    if line:
        print(line.decode('utf-8'))

data: {"id":"chatcmpl-b69846c49246a0c5","object":"chat.completion.chunk","created":1775439033,"model":"deepseek-ai/DeepSeek-R1-Distill-Qwen-32B","choices":[{"index":0,"delta":{"role":"assistant","content":""},"logprobs":null,"finish_reason":null}],"prompt_token_ids":null}
data: {"id":"chatcmpl-b69846c49246a0c5","object":"chat.completion.chunk","created":1775439033,"model":"deepseek-ai/DeepSeek-R1-Distill-Qwen-32B","choices":[{"index":0,"delta":{"reasoning":"Okay"},"logprobs":null,"finish_reason":null,"token_ids":null}]}
data: {"id":"chatcmpl-b69846c49246a0c5","object":"chat.completion.chunk","created":1775439033,"model":"deepseek-ai/DeepSeek-R1-Distill-Qwen-32B","choices":[{"index":0,"delta":{"reasoning":","},"logprobs":null,"finish_reason":null,"token_ids":null}]}
data: {"id":"chatcmpl-b69846c49246a0c5","object":"chat.completion.chunk","created":1775439033,"model":"deepseek-ai/DeepSeek-R1-Distill-Qwen-32B","choices":[{"index":0,"delta":{"reasoning":" I"},"logprobs":null,"finish_reason

In [113]:
res = requests.post(url, json=payload)

is_reasoning = False

for line in res.iter_lines():
    if line:
        decoded = line.decode('utf-8')

        if decoded.startswith('data: '):
            data = decoded.split('data: ')[1]

            if data == '[DONE]':
                break;
            
            json_data = json.loads(data)
            
            tokens = json_data['choices'][0]['delta']

            if tokens.get('reasoning', None):
                if not is_reasoning: 
                    print("<REASONING>")
                    is_reasoning = True
                print(tokens['reasoning'], end='', flush=True)
            
            if tokens.get('content', None):
                if is_reasoning:
                    print("\n<RESPONSE>")
                    is_reasoning = False
                print(tokens['content'], end='', flush=True)

로또 번호 추천기를 파이썬 코드로 작성하는 것은 재미있는 작업입니다. 로또는 일반적으로 6개의 숫자를 뽑는 게임으로, 각 숫자는 1부터 45까지의 범위 내에서 고유한 값이어야 합니다. 파이썬에서는 `random` 모듈을 사용하여 이 작업을 쉽게 수행할 수 있습니다.

아래는 간단한 로또 번호 추천기를 작성하는 파이썬 코드 예시입니다:

```python
import random

def generate_lotto_numbers():
    # 로또 번호는 1부터 45까지의 숫자 중에서 6개를 고르는 것입니다.
    # set을 사용하여 중복을 방지합니다.
    lotto_numbers = set()

    while len(lotto_numbers) < 6:
        # 1부터 45까지의 숫자 중에서 랜덤한 숫자를 선택합니다.
        number = random.randint(1, 45)
        # 선택된 숫자를 set에 추가합니다.
        lotto_numbers.add(number)

    # set은 순서가 없기 때문에, 리스트로 변환하여 정렬합니다.
    lotto_numbers = sorted(list(lotto_numbers))
    return lotto_numbers

# 로또 번호 추천기 실행
def main():
    print("로또 번호 추천기")
    input("로또 번호를 추천받으려면 엔터 키를 누르세요...")
    recommended_numbers = generate_lotto_numbers()
    print("추천된 로또 번호:", recommended_numbers)

if __name__ == "__main__":
    main()
```

위 코드는 다음과 같은 방식으로 작동합니다:

1. `generate_lotto_numbers` 함수는 로또 번호를 생성합니다. 이 함수는 1부터 45까지의 숫자 중에서 6개의 고유한 숫자를 뽑습니다.
2. `set` 자료형을 사용하여 중